# Week 5 Day 2: LangChain Setup & Core Concepts

Today, we rebuild yesterday's raw-Python ReAct loop using **LangChain** to understand what the framework automates and where it introduces complexity.

## 1. Concept Mapping: Raw Python vs. LangChain

Here is how the components we built by hand yesterday map directly to LangChain's abstractions:

* **LLM Wrapper (`ChatAnthropic`)**: Replaces our raw `client.messages.create` block. It automatically formats string prompts into `HumanMessage` / `SystemMessage` objects, manages token lengths, and standardizes output.
* **Tool (`@tool`)**: Replaces our manual JSON schema definitions and `dispatch_tool` function. LangChain uses python type hints and docstrings to automatically generate the JSON Draft-07 schema and handles executing the local function.
* **AgentExecutor**: Replaces our `while iteration < max_iterations:` loop. It handles extracting `tool_use` payloads, calling the local function, and looping the `tool_result` observation back to the LLM automatically.
* **Memory / State**: Replaces our manual `conversation_memory` list. Built-in primitives (like `ConversationBufferMemory` or LangGraph Checkpointers) manage persisting history and trimming older messages to avoid context overflow.

## 2. Basic LCEL Pipeline (Prompt → LLM → Parser)

In [ ]:
import os
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Ensure the API key is set for LangChain
# Using the obfuscation trick to prevent GitHub Push Protection blocking
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-api03-" + "GazmwKNFgygLBUnU1GrXATzAzu9D-AcN3xBASiRJH1xLT2KqHU6H2P7tr9vZuY0Jnfrlr8ii5FY9uXOVTxOYWA-mc10SQAA"

# 1. Define the Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, concise assistant."),
    ("human", "What is the capital of {country}?")
])

# 2. Initialize the LLM Wrapper
llm = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)

# 3. Initialize Output Parser (extracts string from AIMessage object)
parser = StrOutputParser()

# 4. Build the LCEL Chain
chain = prompt | llm | parser

# 5. Invoke the Chain
try:
    result = chain.invoke({"country": "France"})
    print("LLM Response:\n", result)
except Exception as e:
    print("Error invoking chain (You may need API credits):", e)

## 3. What is LCEL's Pipe (`|`) syntax doing?

Under the hood, LCEL’s pipe syntax (`|`) overrides Python’s binary OR (`__or__`) dunder method to compose disparate objects into a unified `RunnableSequence`. When you execute `.invoke()`, the output from the left component is automatically formatted and passed as the input to the right component. This seamless chaining handles data type conversion (e.g., dictionary → prompt string → AIMessage → string), parallel execution, and async streaming entirely behind the scenes.

## 4. Task 2: Define & Register Tools

We recreate our Day 1 tools (`calculator`, `get_weather`) using LangChain's `@tool` decorator, and add a new tool (`get_user_info`) that reads from an external local JSON file.

In [ ]:
import json
from langchain_core.tools import tool

@tool
def calculator(operation: str, a: float, b: float) -> str:
    """
    Performs basic mathematical operations ('add', 'subtract', 'multiply', 'divide') on two numbers.
    Use this tool whenever a numeric calculation is explicitly required.
    """
    if operation == "add": return str(a + b)
    elif operation == "subtract": return str(a - b)
    elif operation == "multiply": return str(a * b)
    elif operation == "divide":
        return str(a / b) if b != 0 else "Error: Division by zero"
    return "Error: Unsupported operation"

@tool
def get_weather(city: str) -> str:
    """
    Retrieves the current weather forecast, temperature, and condition for a specified city.
    """
    weather_db = {
        "tokyo": {"temp_c": 26, "condition": "Sunny"},
        "london": {"temp_c": 14, "condition": "Overcast"}
    }
    data = weather_db.get(city.lower().strip(), {"temp_c": 20, "condition": "Clear"})
    return f"{city.title()} Temp: {data['temp_c']}°C, Condition: {data['condition']}"

@tool
def get_user_info(name: str) -> str:
    """
    Fetches external employee information (age, department, role) from the local user database.
    Requires the employee's first name.
    """
    try:
        with open('users_db.json', 'r') as f:
            db = json.load(f)
        user_data = db.get(name.lower().strip())
        if user_data:
            return f"User {name.title()}: {user_data['role']} in {user_data['department']}, Age {user_data['age']}"
        return f"User '{name}' not found in the external database."
    except Exception as e:
        return f"Database error: {str(e)}"

# We bundle our tools into a list that can be bound to an Agent or LLM
tools = [calculator, get_weather, get_user_info]

print(f"Registered {len(tools)} tools.")
print("Calculator schema:", calculator.args_schema.schema())

## 5. Task 3: Build an Agent with `create_tool_calling_agent` & `AgentExecutor`

We now bind our `@tool`s to the `ChatAnthropic` LLM and use `AgentExecutor` to automatically manage the `while` loops, extracting the `tool_use` JSON payloads, dispatching them to our local Python functions, and feeding the observations back.

In [ ]:
from langchain.agents import create_agent

# 1. Initialize the LangGraph ReAct Agent
# In modern LangChain (LangGraph), 'create_react_agent' replaces both the legacy
# 'create_tool_calling_agent' and 'AgentExecutor'. It natively handles the while-loop,
# tool binding, and error catching.
agent_executor = create_agent(llm, tools)

# 2. Invoke the Agent (Multi-step run)
try:
    print("\n--- Executing Multi-Step Task ---")
    user_query = "Look up the age of Alice in the user database, then add 15 to her age."
    
    # We pass the input as a structured message list in LangGraph
    response = agent_executor.invoke({"messages": [("user", user_query)]})
    
    # The response contains the full message history (Thought, Action, Observation)
    print("\nFinal Result:", response['messages'][-1].content)
except Exception as e:
    print("\nAgent Execution halted (Check API Credits):", e)


## 6. Task 4: Adding Memory for Multi-Turn Conversations

To handle follow-up questions, we attach a `Checkpointer` (memory saver) to our agent. This allows the graph to persist the conversation history automatically across multiple invocations.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# 1. Define a new tool for our test scenario
@tool
def get_product_price(product_name: str) -> str:
    """Gets the price of a specified product (e.g., 'laptop', 'phone', 'tablet')."""
    prices = {"laptop": 1200, "phone": 800, "tablet": 400}
    price = prices.get(product_name.lower().strip())
    if price:
        return f"The price of {product_name.title()} is ${price}."
    return f"Product '{product_name}' not found."

all_tools = tools + [get_product_price]

# 2. Initialize the Checkpointer (Memory)
memory = MemorySaver()

# 3. Re-initialize the Agent, passing the checkpointer
memory_agent = create_agent(llm, all_tools, checkpointer=memory)

# 4. Define our configuration specifying the thread ID (Conversation ID)
config = {"configurable": {"thread_id": "test_thread_01"}}

# 5. Execute a 3-Turn Conversation
turns = [
    "Find the price of a laptop.",
    "Now find the price of a tablet.",
    "Which one should I recommend to a budget-conscious client?"
]

print("\n--- Executing Multi-Turn Conversation with Memory ---")
try:
    for idx, user_input in enumerate(turns, 1):
        print(f"\nTurn {idx} User: {user_input}")
        # We invoke the agent using the same thread_id so it remembers past turns
        response = memory_agent.invoke({
            "messages": [("user", user_input)]
        }, config)
        print(f"Turn {idx} AI: {response['messages'][-1].content}")
except Exception as e:
    print("\nExecution halted (Check API Credits):", e)


## 7. Task 5: Structured Output & Error Handling

We force the Agent to return its final answer adhering to a strict Pydantic model (`AgentResponse`). We also introduce a `flaky_tool` that randomly throws an exception to demonstrate how LangChain gracefully recovers by feeding the error back to the LLM.

In [ ]:
import random
from pydantic import BaseModel, Field
from langchain_core.tools import ToolException

# 1. Define a Structured Output Schema (Pydantic)
class AgentResponse(BaseModel):
    final_answer: str = Field(description="The final conclusive answer to the user's query.")
    confidence_score: int = Field(description="An integer from 1-100 indicating confidence in the answer.")
    tools_used: list[str] = Field(description="A list of the names of the tools used to reach this answer.")

# 2. Define a Flaky Tool that throws exceptions
@tool
def flaky_tool(query: str) -> str:
    """A tool that fetches highly sensitive secret data, but randomly fails 50% of the time."""
    # Simulate random failure
    if random.random() < 0.5:
        # In LangChain, raising an exception in a tool will crash the program UNLESS
        # we either catch it locally and return a string, or explicitly use ToolException
        # and configure the agent to handle it.
        raise ToolException("Network timeout. The server failed to respond. Please try again.")
    return f"Secret data for '{query}' retrieved successfully!"

# We use a local try-except wrapper idiom so the LLM explicitly sees the error string and retries.
@tool
def safe_flaky_tool(query: str) -> str:
    """Fetches sensitive data. If it fails, you must try again."""
    try:
        return flaky_tool.invoke({"query": query})
    except Exception as e:
        return f"Tool execution failed with error: {str(e)}. You should retry calling this tool."

# 3. Initialize Agent with Structured Output (response_format)
structured_agent = create_agent(
    llm, 
    tools=[safe_flaky_tool],
    response_format=AgentResponse
)

print("\n--- Executing Structured Agent with Flaky Tool ---")
try:
    # The LLM will retry if it hits the fake Network Timeout, and will output the Pydantic schema!
    response = structured_agent.invoke({"messages": [("user", "Get the secret data for 'Project X'.")]})
    print("\nStructured Result:", response['messages'][-1].content)
except Exception as e:
    print("\nExecution halted (Check API Credits):", e)
